# Explanation:
mBART is a multilingual sequence-to-sequence model based on the BART architecture.
It is designed to work across multiple languages.
Unlike DistilBART, mBART is not compressed; it is larger and usually slower,
but it can support multilingual input, which is useful for this use case.

Gold layer — mBART summary

Pipeline before model use:
upload pdf -> extract to txt -> clean txt -> preprocess for nlp -> summary prep -> summary -> evaluation



In [133]:
import json
import math
import re
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer


# Configurations

In [134]:
INPUT_JSON = "../../../Data/silver/doc_01_silver_nlp.json"
OUTPUT_DIR = "../../../Data/gold/MBART/"
OUTPUT_FILE = "multilingual_summary_output.json"

DOC_ID_COLUMN = "document_id"
TEXT_COLUMN = "cleaned_text"
SENTENCES_COLUMN = "sentences"
ENTITIES_COLUMN = "entities"

# Supported language routing
# Dutch model: summarization-tuned Dutch checkpoint
# English model: summarization-tuned English checkpoint
# Fallback: multilingual summarization checkpoint
MODEL_CONFIGS = {
    "nl": {
        "model_name": "ml6team/mbart-large-cc25-cnn-dailymail-nl",
        "family": "mbart",
        "src_lang": "nl_XX",
        "tgt_lang": "nl_XX",
        "max_input_len": 1024,
        "default_min_out": 60,
        "default_max_out": 180,
    },
    "en": {
        "model_name": "facebook/bart-large-cnn",
        "family": "bart",
        "src_lang": None,
        "tgt_lang": None,
        "max_input_len": 1024,
        "default_min_out": 60,
        "default_max_out": 180,
    },
    "fallback": {
        "model_name": "csebuetnlp/mT5_multilingual_XLSum",
        "family": "mt5",
        "src_lang": None,
        "tgt_lang": None,
        "max_input_len": 512,
        "default_min_out": 40,
        "default_max_out": 128,
    },
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MAX_SIMILARITY = 0.68
MAX_WINDOW_EXPANSION = 3      # ±2 neighboring sentences
MAX_SELECTED_WINDOWS = 8
MIN_SENTENCE_CHARS = 30
MIN_SENTENCE_WORDS = 8
MAX_KEY_ENTITIES = 15

SUMMARY_RATIO = 0.04  # 4% of input tokens, with min/max caps
WORDS_TO_TOKENS = 1.3
MIN_SUMMARY_TOKENS = 40
MAX_SUMMARY_TOKENS = 360



# Model caching

In [135]:
_MODEL_CACHE: Dict[str, Tuple[Any, Any]] = {}

# Basic Helpers

In [136]:
SECTION_LABELS = {
    "abstract", "introduction", "background", "discussion", "conclusion",
    "conclusions", "results", "result", "method", "methods", "methodology",
    "references", "bibliography", "appendix", "summary", "contents",
    "table of contents"
}

def remove_editorial_artifacts(text: str) -> str:
    text = re.sub(r"Commented\s*\[[^\]]+\]:.*?(?=\s[A-Z]|\Z)", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"Tracked Changes?.*?(?=\s[A-Z]|\Z)", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"\bPage\s+\d+\s+of\s+\d+\b", " ", text, flags=re.IGNORECASE)
    return normalize_whitespace(text)

def safe_string(value: Any) -> str:
    return "" if value is None else str(value)


def normalize_whitespace(text: Any) -> str:
    return re.sub(r"\s+", " ", safe_string(text)).strip()


def simple_tokenize(text: Any) -> List[str]:
    return re.findall(r"\b\w+\b", safe_string(text).lower())


def dedupe_keep_order(items: List[str]) -> List[str]:
    seen = set()
    out = []
    for item in items:
        clean = normalize_whitespace(item)
        key = clean.lower()
        if clean and key not in seen:
            seen.add(key)
            out.append(clean)
    return out

def window_quality_score(sentences: List[str]) -> float:
    if not sentences:
        return 0.0

    good = 0
    bad = 0

    for s in sentences:
        if looks_like_reference(s) or looks_like_external_example(s) or is_heading_like(s):
            bad += 1
        else:
            good += 1

    total = good + bad
    if total == 0:
        return 0.0

    return good / total

def raw_window_quality_score(raw_sentences: List[str]) -> float:
    if not raw_sentences:
        return 0.0

    good = 0
    bad = 0

    for s in raw_sentences:
        cleaned = clean_sentence_for_selection(s)
        if (
            looks_like_reference(cleaned)
            or looks_like_external_example(cleaned)
            or is_heading_like(cleaned)
            or is_table_of_contents_line(cleaned)
        ):
            bad += 1
        else:
            good += 1

    total = good + bad
    return good / total if total > 0 else 0.0

# Language detection (NL / EN)

In [137]:
DUTCH_HINTS = {
    "de", "het", "een", "van", "voor", "met", "niet", "dat", "op", "aan",
    "als", "bij", "door", "over", "ook", "meer", "dan", "tussen", "wel",
    "wordt", "werden", "onderzoek", "resultaten", "samenvatting"
}

ENGLISH_HINTS = {
    "the", "and", "for", "with", "this", "that", "from", "into", "between",
    "more", "than", "results", "study", "paper", "summary", "research",
    "using", "shows", "indicates", "overall"
}


def detect_language(text: str) -> str:
    """
    Lightweight heuristic language detector for Dutch vs English.
    Falls back to English if inconclusive.
    """
    tokens = simple_tokenize(text[:4000])

    if not tokens:
        return "en"

    dutch_score = sum(1 for t in tokens if t in DUTCH_HINTS)
    english_score = sum(1 for t in tokens if t in ENGLISH_HINTS)

    # Very lightweight morphology hints
    joined = " ".join(tokens[:500])

    if re.search(r"\b(ij|sch|lijk|heid|ingen)\b", joined):
        dutch_score += 2
    if re.search(r"\b(the|this|that|which|their|using)\b", joined):
        english_score += 2

    if dutch_score > english_score + 2:
        return "nl"
    return "en"

    




# Sentence cleaning and selection

In [138]:

def remove_section_prefix(sentence: str) -> str:
    s = normalize_whitespace(sentence)
    s = re.sub(
        r"^(abstract|introduction|background|discussion|conclusion|conclusions|results|result|method|methods|methodology|summary|samenvatting|inleiding|resultaten|conclusie)\b[:\-\s]*",
        "",
        s,
        flags=re.IGNORECASE
    )
    return normalize_whitespace(s)


def is_table_of_contents_line(line: str) -> bool:
    s = normalize_whitespace(line)

    if "table of contents" in s.lower():
        return True
    if "inhoudsopgave" in s.lower():
        return True
    if re.search(r"\.{4,}\s*\d+$", s):
        return True
    if re.search(r"\.{4,}", s):
        return True
    if re.match(r"^\d+(\.\d+)*\s+.+\s+\d+$", s.lower()):
        return True
    if re.search(r"\b\d+\.\d+\b", s) and len(s) < 120:
        return True

    return False


def looks_like_reference(sentence: str) -> bool:
    s = normalize_whitespace(sentence)

    # Classic reference markers
    if re.search(r"\bet al\.", s, flags=re.IGNORECASE):
        return True
    if re.search(r"\bdoi\b", s, flags=re.IGNORECASE):
        return True
    if re.search(r"https?://|www\.", s):
        return True
    if re.search(r"\breferences\b|\bbibliography\b|\bliteratuur\b|\bbronvermelding\b", s, flags=re.IGNORECASE):
        return True
    if re.search(r"\(\s*[A-Z][A-Za-z].*\d{4}.*\)", s):
        return True

    # Citation / footnote markers
    if re.search(r"\[\d+\]", s):
        return True
    if re.search(r"\(\d{4}\)", s):
        return True
    if re.search(r"\bvol\.\b|\bno\.\b|\bpp\.\b", s, flags=re.IGNORECASE):
        return True

    # Source attribution style
    if re.search(r"\baccording to\b|\breported by\b|\bsource:\b", s, flags=re.IGNORECASE):
        return True
    if re.search(r"\bvolgens\b|\bbron:\b", s, flags=re.IGNORECASE):
        return True

    # Date-heavy news/report style sentences
    if re.search(r"\b(january|february|march|april|may|june|july|august|september|october|november|december)\b", s, flags=re.IGNORECASE):
        if re.search(r"\b\d{4}\b", s):
            return True
    if re.search(r"\b(januari|februari|maart|april|mei|juni|juli|augustus|september|oktober|november|december)\b", s, flags=re.IGNORECASE):
        if re.search(r"\b\d{4}\b", s):
            return True

    # Too many capitalized entities often means citation/example/news snippet
    capitalized_words = re.findall(r"\b[A-Z][a-z]+\b", s)
    if len(capitalized_words) >= 6 and len(s.split()) < 40:
        return True

    return False

def looks_like_external_example(sentence: str) -> bool:
    s = normalize_whitespace(sentence)

    trigger_patterns = [
        r"\bfor example\b",
        r"\bfor instance\b",
        r"\bsuch as\b",
        r"\bcase study\b",
        r"\bin the case of\b",
        r"\be.g.\b",
        r"\bbijvoorbeeld\b",
        r"\bzoals\b",
        r"\bcasus\b",
    ]

    external_org_patterns = [
        r"\bUnited Nations\b",
        r"\bOCHA\b",
        r"\bU\.N\.\b",
        r"\bDepartment of Education\b",
        r"\bAmerican Institute of Architects\b",
        r"\bAssociation for Computing Machinery\b",
        r"\bRoyal College of Music\b",
        r"\bRoyal Institute of Technology\b",
        r"\bACM\b",
        r"\bAIA\b",
    ]

    if any(re.search(p, s, flags=re.IGNORECASE) for p in trigger_patterns):
        return True

    if any(re.search(p, s, flags=re.IGNORECASE) for p in external_org_patterns):
        return True

    return False

def is_heading_like(sentence: str) -> bool:
    s = normalize_whitespace(sentence)

    if not s:
        return True
    if s.lower() in SECTION_LABELS:
        return True
    if len(s.split()) <= 5 and s.istitle():
        return True
    if re.fullmatch(r"[A-Z][A-Za-zÀ-ÿ\s/&\-]{1,80}", s) and len(s.split()) <= 8:
        return True
    if re.match(r"^\d+(\.\d+)*\s+[A-Za-zÀ-ÿ]", s):
        return True

    return False


def clean_sentence_for_selection(sentence: str) -> str:
    s = remove_section_prefix(sentence)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def is_good_sentence(sentence: str) -> bool:
    s = clean_sentence_for_selection(sentence)

    if len(s) < MIN_SENTENCE_CHARS:
        return False
    if len(s.split()) < MIN_SENTENCE_WORDS:
        return False
    if is_table_of_contents_line(s):
        return False
    if looks_like_reference(s):
        return False
    if looks_like_external_example(s):
        return False
    if is_heading_like(s):
        return False
    if re.match(r"^\d+(\.\d+)*\s*$", s):
        return False
    if "................................................................" in s:
        return False
    if "•" in s and len(s.split()) < 15:
        return False

    return True



# Scoring

In [139]:
def jaccard_similarity(a: str, b: str) -> float:
    set_a = set(simple_tokenize(a))
    set_b = set(simple_tokenize(b))
    if not set_a or not set_b:
        return 0.0
    return len(set_a & set_b) / len(set_a | set_b)


def sentence_position_score(index: int, total: int) -> float:
    if total <= 0:
        return 0.0

    relative = index / max(total, 1)

    if relative < 0.10:
        return 0.25
    if relative < 0.25:
        return 0.15
    if relative > 0.90:
        return 0.20

    return 0.0


def sentence_information_score(sentence: str) -> float:
    s = sentence.lower()
    word_count = len(sentence.split())
    score = 0.0

    if 12 <= word_count <= 35:
        score += 1.0
    elif 36 <= word_count <= 50:
        score += 0.6
    elif word_count > 55:
        score -= 0.5

    discourse_terms = [
        "this study", "this report", "this paper", "the aim", "the purpose",
        "the objective", "the analysis", "the findings", "the results",
        "shows that", "demonstrates that", "indicates that", "suggests that",
        "because", "therefore", "however", "overall", "in conclusion",
        "dit onderzoek", "dit rapport", "de resultaten", "de bevindingen",
        "toont aan", "laat zien", "wijst erop", "daarom", "echter",
        "concluderend", "samengevat"
    ]
    for term in discourse_terms:
        if term in s:
            score += 0.5

    tokens = simple_tokenize(sentence)
    if tokens:
        unique_ratio = len(set(tokens)) / len(tokens)
        score += 0.8 * unique_ratio

    has_substance = any(x in s for x in [
        "shows", "demonstrates", "indicates", "suggests", "reveals",
        "therefore", "however", "because", "although", "despite",
        "in conclusion", "overall", "result", "finding", "impact",
        "toont", "laat zien", "wijst", "resultaat", "bevinding", "effect"
    ])
    is_intro_like = (
        word_count < 25
        and re.match(r"^(we |this |in this |wij |dit |in dit )", s)
        and not has_substance
    )
    if is_intro_like:
        score -= 0.8
        # Penalty for date-heavy / news-like factual snippets
    if re.search(r"\b\d{4}\b", s):
        score -= 0.25

    if re.search(r"\b(january|february|march|april|may|june|july|august|september|october|november|december)\b", s):
        score -= 0.4

    if re.search(r"\b(januari|februari|maart|april|mei|juni|juli|augustus|september|oktober|november|december)\b", s):
        score -= 0.4
        
    return score


def rank_sentences(sentences: List[str]) -> List[Tuple[int, str, float]]:
    cleaned_candidates = []
    for i, s in enumerate(sentences):
        cleaned = clean_sentence_for_selection(s)
        if is_good_sentence(cleaned):
            cleaned_candidates.append((i, cleaned))

    total = len(cleaned_candidates)
    scored = []

    for original_idx, sentence in cleaned_candidates:
        score = sentence_information_score(sentence) + sentence_position_score(original_idx, total)
        scored.append((original_idx, sentence, score))

    scored.sort(key=lambda x: x[2], reverse=True)
    return scored



# Adjacent sentence window extraction

In [140]:
@dataclass
class SentenceWindow:
    start: int
    end: int
    score: float


def build_sentence_windows(
    sentences: List[str],
    expansion: int = MAX_WINDOW_EXPANSION,
    max_windows: int = MAX_SELECTED_WINDOWS
) -> List[SentenceWindow]:
    ranked = rank_sentences(sentences)
    if not ranked:
        return []

    total_sentences = len(sentences)

    def bucket(idx: int) -> str:
        rel = idx / max(total_sentences, 1)
        if rel < 0.33:
            return "early"
        if rel < 0.66:
            return "middle"
        return "late"

    bucket_targets = {"early": [], "middle": [], "late": []}
    for idx, _, score in ranked:
        bucket_targets[bucket(idx)].append((idx, score))

    chosen_windows = []
    used_ranges = []

    # first pass: guarantee coverage
    for part in ["early", "middle", "late"]:
        for idx, score in bucket_targets[part]:
            start = max(0, idx - expansion)
            end = min(len(sentences) - 1, idx + expansion)

            overlaps = any(not (end < s or start > e) for s, e in used_ranges)
            if overlaps:
                continue

            chosen_windows.append(SentenceWindow(start=start, end=end, score=score))
            used_ranges.append((start, end))
            break

    # second pass: fill remaining slots with best remaining windows
    for idx, _, score in ranked:
        if len(chosen_windows) >= max_windows:
            break

        start = max(0, idx - expansion)
        end = min(len(sentences) - 1, idx + expansion)

        overlaps = any(not (end < s or start > e) for s, e in used_ranges)
        if overlaps:
            continue

        chosen_windows.append(SentenceWindow(start=start, end=end, score=score))
        used_ranges.append((start, end))

    chosen_windows.sort(key=lambda w: (w.start, w.end))

    merged = []
    for w in chosen_windows:
        if not merged:
            merged.append(w)
            continue

        prev = merged[-1]
        if w.start <= prev.end + 1:
            merged[-1] = SentenceWindow(
                start=prev.start,
                end=max(prev.end, w.end),
                score=max(prev.score, w.score)
            )
        else:
            merged.append(w)

    return merged


def load_model_and_tokenizer(language: str):
    config = MODEL_CONFIGS.get(language, MODEL_CONFIGS["fallback"])
    model_name = config["model_name"]

    if model_name in _MODEL_CACHE:
        return _MODEL_CACHE[model_name]

    tokenizer_kwargs = {}
    if config["family"] == "mbart" and config["src_lang"]:
        tokenizer_kwargs["src_lang"] = config["src_lang"]
        tokenizer_kwargs["use_fast"] = False

    tokenizer = AutoTokenizer.from_pretrained(model_name, **tokenizer_kwargs)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(DEVICE)
    model.eval()

    _MODEL_CACHE[model_name] = (tokenizer, model)
    return tokenizer, model


def build_summary_input_from_windows(
    sentences: List[str],
    language: str,
    max_input_tokens: Optional[int] = None
) -> Tuple[str, List[Dict[str, Any]]]:
    """
    Build coherent model input from merged adjacent windows,
    preserving original order and local context.
    """
    config = MODEL_CONFIGS.get(language, MODEL_CONFIGS["fallback"])
    tokenizer, _ = load_model_and_tokenizer(language)
    max_len = max_input_tokens or config["max_input_len"]

    windows = build_sentence_windows(sentences)
    if not windows:
        return "", []

    selected_chunks = []
    window_debug = []

    for window in windows:
        raw_window_sentences = sentences[window.start:window.end + 1]

        if raw_window_quality_score(raw_window_sentences) < 0.55:
            continue

        block_sentences = []
        for i in range(window.start, window.end + 1):
            s = clean_sentence_for_selection(sentences[i])
            if not is_good_sentence(s):
                continue
            if not re.search(r"[.!?…]$", s.strip()):
                continue
            block_sentences.append(s)

        if not block_sentences:
            continue

        block_text = " ".join(block_sentences).strip()
        if not block_text:
            continue
        
        if window_quality_score(block_sentences) < 0.55:
            continue
        
        candidate = " ".join(selected_chunks + [block_text]).strip()
        token_count = len(tokenizer(candidate, truncation=False)["input_ids"])

        if token_count > max_len - 20:
            # Try to fit the block sentence-by-sentence
            partial = []
            for sent in block_sentences:
                test_candidate = " ".join(selected_chunks + partial + [sent]).strip()
                test_tokens = len(tokenizer(test_candidate, truncation=False)["input_ids"])
                if test_tokens > max_len - 20:
                    break
                partial.append(sent)

            if partial:
                fitted_text = " ".join(partial).strip()
                selected_chunks.append(fitted_text)
                window_debug.append({
                    "start": window.start,
                    "end": min(window.start + len(partial) - 1, window.end),
                    "score": round(window.score, 4),
                    "text": fitted_text
                })
            break

        selected_chunks.append(block_text)
        window_debug.append({
            "start": window.start,
            "end": window.end,
            "score": round(window.score, 4),
            "text": block_text
        })

    final_text = normalize_whitespace(" ".join(selected_chunks))
    return final_text, window_debug




# Length control

In [141]:
def compute_output_length_from_document(original_word_count: int) -> Tuple[int, int]:
    if original_word_count <= 0:
        return 40, 100

    target_words = max(1, int(original_word_count * SUMMARY_RATIO))
    target_tokens = int(target_words * WORDS_TO_TOKENS)

    min_out = max(30, int(target_tokens * 0.55))
    max_out = max(min_out + 20, int(target_tokens * 0.85))

    min_out = min(min_out, 120)
    max_out = min(max_out, 220)

    return min_out, max_out

# Summary creation

In [142]:
def summarize_text(
    text: str,
    language: str,
    original_word_count: int
) -> Tuple[str, Dict[str, Any]]:
    config = MODEL_CONFIGS.get(language, MODEL_CONFIGS["fallback"])
    tokenizer, model = load_model_and_tokenizer(language)

    input_ids = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=config["max_input_len"]
    )
    input_token_count = int(input_ids["input_ids"].shape[1])
    input_ids = {k: v.to(DEVICE) for k, v in input_ids.items()}

    min_out, max_out = compute_output_length_from_document(original_word_count)
    
    generate_kwargs = {
        "max_length": max_out,
        "min_length": min_out,
        "num_beams": 5,
        "length_penalty": 1.0,
        "no_repeat_ngram_size": 3,
        "early_stopping": True,
    }

    if config["family"] == "mbart" and config["tgt_lang"]:
        tokenizer.tgt_lang = config["tgt_lang"]
        if hasattr(tokenizer, "lang_code_to_id"):
            generate_kwargs["forced_bos_token_id"] = tokenizer.lang_code_to_id[config["tgt_lang"]]

    with torch.no_grad():
        summary_ids = model.generate(
            input_ids["input_ids"],
            attention_mask=input_ids.get("attention_mask"),
            **generate_kwargs
        )

    output_token_count = int(summary_ids.shape[1])
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    summary = clean_generated_summary(summary)

    meta = {
        "model_name": config["model_name"],
        "model_family": config["family"],
        "input_token_count_to_model": input_token_count,
        "output_token_count_from_model": output_token_count,
        "target_min_tokens": min_out,
        "target_max_tokens": max_out,
        "original_word_count": original_word_count,
    }
    return summary, meta


def clean_generated_summary(summary: str) -> str:
    summary = re.sub(
        r"^(summarize the following document|summary|samenvatting)[:\s-]*",
        "",
        summary,
        flags=re.IGNORECASE
    ).strip()
    summary = remove_section_prefix(summary)
    summary = re.sub(r"\s+", " ", summary).strip()

    # Light cleanup for repeated punctuation
    summary = re.sub(r"\s+([,.;:!?])", r"\1", summary)
    summary = re.sub(r"([.?!]){2,}", r"\1", summary)

    return summary



# Evaluation

In [143]:
def get_key_entities_from_row(row: pd.Series, max_entities: int = MAX_KEY_ENTITIES) -> List[str]:
    entities = row.get(ENTITIES_COLUMN, [])

    extracted = []
    if isinstance(entities, list):
        for ent in entities:
            if isinstance(ent, dict):
                label = ent.get("label", "")
                text = normalize_whitespace(ent.get("text", ""))
                if label in {"PERSON", "ORG", "GPE", "DATE", "PRODUCT", "EVENT"} \
                        and len(text) > 2 \
                        and len(text.split()) <= 6:
                    extracted.append(text)

    return dedupe_keep_order(extracted)[:max_entities]


def evaluate_summary(
    original_text: str,
    summary_text: str,
    source_entities: List[str]
) -> Dict[str, Any]:
    original_tokens = simple_tokenize(original_text)
    summary_tokens = simple_tokenize(summary_text)

    original_len = len(original_tokens)
    summary_len = len(summary_tokens)
    compression_ratio = summary_len / original_len if original_len > 0 else 0.0

    summary_lower = summary_text.lower()

    preserved_entities = []
    missed_entities = []

    for ent in source_entities:
        ent_clean = normalize_whitespace(ent)
        if ent_clean.lower() in summary_lower:
            preserved_entities.append(ent_clean)
        else:
            missed_entities.append(ent_clean)

    total_entities = len(source_entities)
    entity_preservation_score = len(preserved_entities) / total_entities if total_entities > 0 else None

    return {
        "original_token_count": original_len,
        "summary_token_count": summary_len,
        "compression_ratio": compression_ratio,
        "preserved_entities": preserved_entities,
        "missed_entities": missed_entities,
        "entity_preservation_score": entity_preservation_score
    }


def sentence_coverage_score(source_sentences: List[str], summary_text: str, threshold: float = 0.2) -> float:
    summary_words = set(simple_tokenize(summary_text))
    covered = 0

    for sent in source_sentences:
        sent_words = set(simple_tokenize(sent))
        if not sent_words:
            continue
        overlap = len(summary_words & sent_words) / len(sent_words)
        if overlap >= threshold:
            covered += 1

    return covered / len(source_sentences) if source_sentences else 0.0


def summary_redundancy_score(summary_text: str) -> float:
    summary_sentences = re.split(r"(?<=[.!?])\s+", normalize_whitespace(summary_text))

    if len(summary_sentences) < 2:
        return 0.0

    similarities = []
    for i in range(len(summary_sentences)):
        for j in range(i + 1, len(summary_sentences)):
            similarities.append(jaccard_similarity(summary_sentences[i], summary_sentences[j]))

    return sum(similarities) / len(similarities) if similarities else 0.0



# Row summary

In [144]:
def summarize_document_row(row: pd.Series) -> Dict[str, Any]:
    document_id = safe_string(row.get(DOC_ID_COLUMN, ""))
    text = normalize_whitespace(row.get(TEXT_COLUMN, ""))

    if not text:
        raise ValueError("Input document has no usable cleaned_text.")

    sentences = row.get(SENTENCES_COLUMN, [])
    if not isinstance(sentences, list) or not sentences:
        raise ValueError("Input document has no usable sentences from silver_nlp.")

    sentences = [normalize_whitespace(s) for s in sentences if normalize_whitespace(s)]

    language = detect_language(text)
    print(f"Detected language: {language}")

    summary_input, selected_windows = build_summary_input_from_windows(
        sentences=sentences,
        language=language
    )

    if not summary_input:
        fallback_sentences = [
            clean_sentence_for_selection(s)
            for s in sentences
            if is_good_sentence(s)
            and re.search(r"[.!?…]$", clean_sentence_for_selection(s))
        ]
        summary_input = normalize_whitespace(" ".join(fallback_sentences[:12]))

    if not summary_input:
        summary_input = normalize_whitespace(text[:2000])

    key_entities = get_key_entities_from_row(row)
    original_word_count = len(simple_tokenize(text))

    start = time.time()
    summary, model_meta = summarize_text(
        summary_input,
        language=language,
        original_word_count=original_word_count
    )
    runtime_seconds = time.time() - start

    eval_result = evaluate_summary(
        original_text=text,
        summary_text=summary,
        source_entities=key_entities
    )
    eval_result["runtime_seconds"] = runtime_seconds
    eval_result["sentence_coverage_score"] = sentence_coverage_score(sentences, summary)
    eval_result["summary_redundancy_score"] = summary_redundancy_score(summary)
    eval_result.update(model_meta)

    return {
        "document_id": document_id,
        "language_detected": language,
        "model": model_meta["model_name"],
        "selected_windows": selected_windows,
        "summary_input": summary_input,
        "summary": summary,
        "evaluation": eval_result
    }

# Run main doc

In [145]:
def main() -> None:
    df = pd.read_json(INPUT_JSON, orient="records")

    if df.empty:
        raise ValueError("Input JSON is empty.")

    row = df.loc[0]
    result = summarize_document_row(row)

    output_dir = Path(OUTPUT_DIR)
    output_dir.mkdir(parents=True, exist_ok=True)

    output_path = output_dir / OUTPUT_FILE
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=4, ensure_ascii=False)

    print(f"Loaded device: {DEVICE}")
    print(f"Saved summary output to {output_path}")
    print("\nSUMMARY:\n")
    print(result["summary"])
    print("\nEVALUATION:\n")
    print(json.dumps(result["evaluation"], indent=4, ensure_ascii=False))
    print("\nSUMMARY INPUT:\n")
    print(result["summary_input"])

if __name__ == "__main__":
    main()

Detected language: en


Loading weights: 100%|██████████| 511/511 [00:00<00:00, 12163.96it/s]


Loaded device: cpu
Saved summary output to ..\..\..\Data\gold\MBART\multilingual_summary_output.json

SUMMARY:

In this chapter we will go over the outcomes of the research activities described in the methodological section. Each sub-question will be addressed using the AWS services, we will make use of benchmarks, observation, and evaluations. The results will be shown using tables, images, screenshots, descriptive text, and coding where it’s needed.

EVALUATION:

{
    "original_token_count": 1957,
    "summary_token_count": 54,
    "compression_ratio": 0.027593254982115484,
    "preserved_entities": [
        "AWS"
    ],
    "missed_entities": [
        "HAREMAN",
        "HZ University of Applied Sciences",
        "Installation &",
        "WFA",
        "CloudWatch",
        "• Medium",
        "• Large",
        "dataset_large.txt",
        "~480MB",
        "Gutenberg",
        "n.d.",
        "WSL",
        "UML",
        "Diagram"
    ],
    "entity_preservation_score": 0.06